In [1]:
import pandas as pd

# Load raw data fresh — this notebook is self-contained and does not
# depend on variables from 01_initial_inspection.ipynb
df = pd.read_csv("../data/raw/a_steam_data_2021_2025.csv")

print("Loaded shape:", df.shape)

Loaded shape: (65521, 10)


In [2]:
# Monetization flag derived strictly from price (see docs/01_ask.md —
# the 'Free To Play' genre tag was found unreliable for this purpose)
df["is_free"] = df["price"] == 0

print(df["is_free"].value_counts())

is_free
False    53559
True     11962
Name: count, dtype: int64


In [3]:
# Attempt to parse release_date; rows that fail are known to be
# unreleased titles with approximate dates (e.g., "Q4 2025") — confirmed
# in docs/02_prepare.md via the collection script review
df["release_date_parsed"] = pd.to_datetime(df["release_date"], errors="coerce")

# Explicit flag: does this row have a real, parseable exact date?
df["has_exact_release_date"] = df["release_date_parsed"].notna()

print("Rows with exact release date:", df["has_exact_release_date"].sum())
print("Rows without exact release date:", (~df["has_exact_release_date"]).sum())

Rows with exact release date: 64121
Rows without exact release date: 1400


In [4]:
# Per project decision: keep only the parsed date for downstream use.
# The raw text is discarded because 'has_exact_release_date' already
# communicates the same limitation without duplicating audit-only data.
df["release_date"] = df["release_date_parsed"]
df = df.drop(columns=["release_date_parsed"])

print(df["release_date"].dtype)
print(df[["release_year", "release_date", "has_exact_release_date"]].head(3))

datetime64[ns]
   release_year release_date  has_exact_release_date
0          2024   2024-07-05                    True
1          2025   2025-07-25                    True
2          2025   2025-06-17                    True


In [5]:
# Replace nulls with an explicit "Unknown" label rather than leaving
# them as NaN — this keeps missing data visible in Power BI visuals
# instead of silently disappearing from groupings/filters.
cols_to_fill = ["genres", "categories", "developer", "publisher"]

for col in cols_to_fill:
    missing_before = df[col].isna().sum()
    df[col] = df[col].fillna("Unknown")
    print(f"{col}: filled {missing_before} missing values")

genres: filled 66 missing values
categories: filled 7 missing values
developer: filled 53 missing values
publisher: filled 183 missing values


In [6]:
# Build a bridge table (appid, genre) — one row per game-genre
# combination. This resolves the many-to-many relationship for Power BI
# star-schema modeling (see docs/01_ask.md and the Process rationale).
genres_bridge = df[["appid", "genres"]].copy()

# Split the semicolon-delimited string into a list, then explode into rows
genres_bridge["genre"] = genres_bridge["genres"].str.split(";")
genres_bridge = genres_bridge.explode("genre")

# Strip whitespace to avoid duplicate-but-different category values
genres_bridge["genre"] = genres_bridge["genre"].str.strip()

# Keep only the bridge columns
genres_bridge = genres_bridge[["appid", "genre"]]

print("Bridge table shape:", genres_bridge.shape)
print("Unique genres found:", genres_bridge["genre"].nunique())
genres_bridge.head(10)

Bridge table shape: (190626, 2)
Unique genres found: 23


,appid,genre
0,3057270,Action
0,3057270,Adventure
0,3057270,Indie
0,3057270,RPG
0,3057270,Strategy
1,3822840,Casual
1,3822840,Indie
1,3822840,Simulation
1,3822840,Strategy
2,3216640,Adventure


In [7]:
# Sanity check: list all unique genre values to visually inspect
# for inconsistencies (casing, stray whitespace, unexpected values)
print(sorted(genres_bridge["genre"].unique()))

['Accounting', 'Action', 'Adventure', 'Animation & Modeling', 'Audio Production', 'Casual', 'Design & Illustration', 'Early Access', 'Education', 'Free To Play', 'Game Development', 'Indie', 'Massively Multiplayer', 'RPG', 'Racing', 'Simulation', 'Software Training', 'Sports', 'Strategy', 'Unknown', 'Utilities', 'Video Production', 'Web Publishing']


In [8]:
# Analytical scoping decision (NOT a data correction):
# The Steam 'genres' field mixes true gameplay genres with software
# categories (e.g., "Accounting", "Utilities") and product status/model
# tags (e.g., "Early Access", "Free To Play"). This is a known
# characteristic of Steam's own store taxonomy, not a collection error.
#
# For this project's genre dimension — used to answer the business
# question about gameplay genre vs. monetization — only true gameplay
# genres are kept. This filtering applies ONLY to the bridge table below.
# The original 'genres' column in games_clean.csv remains untouched.
GAMEPLAY_GENRES = [
    "Action", "Adventure", "Casual", "Indie", "RPG",
    "Racing", "Simulation", "Sports", "Strategy",
    "Massively Multiplayer"
]

genres_bridge = genres_bridge[
    genres_bridge["genre"].isin(GAMEPLAY_GENRES) | (genres_bridge["genre"] == "Unknown")
]

print("Bridge shape after gameplay-genre filtering:", genres_bridge.shape)
print("Remaining unique genres:", sorted(genres_bridge["genre"].unique()))

Bridge shape after gameplay-genre filtering: (175562, 2)
Remaining unique genres: ['Action', 'Adventure', 'Casual', 'Indie', 'Massively Multiplayer', 'RPG', 'Racing', 'Simulation', 'Sports', 'Strategy', 'Unknown']


In [9]:
# Identify appids that lost ALL genre rows after filtering — these had
# only software/status tags (e.g., a game tagged only "Utilities").
# They must be re-added with "Unknown" so they remain visible in any
# genre-based count, rather than silently disappearing.
all_appids = df["appid"]
appids_with_gameplay_genre = genres_bridge.loc[
    genres_bridge["genre"] != "Unknown", "appid"
].unique()

appids_missing_gameplay_genre = all_appids[~all_appids.isin(appids_with_gameplay_genre)]

# Remove any pre-existing "Unknown" rows for these appids to avoid duplicates,
# then add exactly one "Unknown" row per affected appid
genres_bridge = genres_bridge[
    ~((genres_bridge["appid"].isin(appids_missing_gameplay_genre)) & (genres_bridge["genre"] == "Unknown"))
]

fallback_rows = pd.DataFrame({
    "appid": appids_missing_gameplay_genre.unique(),
    "genre": "Unknown"
})

genres_bridge = pd.concat([genres_bridge, fallback_rows], ignore_index=True)

print("Games with no gameplay genre (assigned 'Unknown'):", len(appids_missing_gameplay_genre.unique()))
print("Final bridge shape:", genres_bridge.shape)
print("Final unique genres:", sorted(genres_bridge["genre"].unique()))

Games with no gameplay genre (assigned 'Unknown'): 195
Final bridge shape: (175691, 2)
Final unique genres: ['Action', 'Adventure', 'Casual', 'Indie', 'Massively Multiplayer', 'RPG', 'Racing', 'Simulation', 'Sports', 'Strategy', 'Unknown']


In [10]:
import os

os.makedirs("../data/processed", exist_ok=True)

# Main fact table — one row per game, source-faithful genres column preserved
df.to_csv("../data/processed/games_clean.csv", index=False)

# Analytical bridge table — many-to-many gameplay genre dimension
genres_bridge.to_csv("../data/processed/games_genres_bridge.csv", index=False)

print("games_clean.csv shape:", df.shape)
print("games_genres_bridge.csv shape:", genres_bridge.shape)

games_clean.csv shape: (65521, 12)
games_genres_bridge.csv shape: (175691, 2)
